# ICS1512 Machine Learning Algorithms Laboratory
## Experiment No. 1 — Exploratory Data Analysis on the Spambase Dataset

**Aim:** Explore NumPy, Pandas, SciPy, Scikit-learn, Matplotlib/Seaborn and perform EDA on the Spambase dataset, including data cleaning, duplicate removal, IQR outlier detection/removal, visualization and correlation analysis.

> **Dataset expected:** `spambase csv.csv` in the same folder as this notebook.


## 1. Libraries Used

- **NumPy:** numerical operations and arrays.
- **Pandas:** loading, cleaning and analyzing tabular data.
- **SciPy:** scientific/statistical utilities.
- **Scikit-learn:** machine-learning utilities; useful for the downstream classification task.
- **Matplotlib / Seaborn:** visualization and correlation plots.

The PDF's experiment specifically focuses on EDA rather than training a spam classifier.


In [1]:
# Import the libraries used in the experiment
import numpy as np
import pandas as pd
import scipy
import sklearn
import matplotlib.pyplot as plt
import seaborn as sns

print("NumPy version:", np.__version__)
print("Pandas version:", pd.__version__)
print("SciPy version:", scipy.__version__)
print("Scikit-learn version:", sklearn.__version__)


NumPy version: 2.3.3
Pandas version: 2.3.2
SciPy version: 1.16.2
Scikit-learn version: 1.7.2


## 2. Load the Spambase Dataset

In [2]:
# Keep the CSV file in the same directory as this notebook.
file_path = "spambase csv.csv"
df = pd.read_csv(file_path)

print("=" * 60)
print("EXPLORATORY DATA ANALYSIS")
print("=" * 60)

print("\nFirst 5 Rows")
display(df.head())

print("\nDataset Information")
df.info()

print("\nDataset Shape:", df.shape)

print("\nColumn Names")
print(df.columns.tolist())


FileNotFoundError: [Errno 2] No such file or directory: 'spambase csv.csv'

### What this section means

The PDF reports **4,601 rows and 58 columns**. The first 57 columns are numerical email-related features and the last column, `class`, is the target: `0` for non-spam and `1` for spam. The PDF reports 55 `float64` columns and 3 `int64` columns. fileciteturn0file0L34-L54 fileciteturn0file0L96-L102

The `word_freq_*` columns represent word-frequency features, `char_freq_*` columns represent character-frequency features, and the `capital_run_length_*` columns describe capital-letter runs.


## 3. Missing-Value Analysis and Cleaning

In [ ]:
print("Missing Values Before Cleaning")
print(df.isnull().sum())

# Remove rows containing missing values.
df = df.dropna()

print("\nMissing Values After Cleaning")
print(df.isnull().sum())


### What this means

Missing-value checking asks: **“Are any cells empty/unknown?”**

In the PDF, all 58 columns already had **0 missing values**, both before and after cleaning. Therefore, the `dropna()` step does not reduce the row count. fileciteturn0file0L116-L119

**Faculty explanation:** “I checked for missing values first. Since there were no missing values, no actual rows were removed by missing-value cleaning.”


## 4. Duplicate-Row Removal

In [ ]:
duplicate_count = df.duplicated().sum()
print("Duplicate Rows:", duplicate_count)

df = df.drop_duplicates().reset_index(drop=True)

print("Shape After Removing Duplicates:", df.shape)
print("\nData Types")
print(df.dtypes.value_counts())


### What this means

A **duplicate row** is a complete row that is repeated in the dataset. Keeping duplicates can give repeated observations extra influence.

The PDF reports **391 duplicate rows**, reducing the dataset from **4,601 × 58** to **4,210 × 58**. fileciteturn0file0L120-L124

So:

**4,601 − 391 = 4,210 rows**


## 5. Unique Values and Numerical Statistics

In [ ]:
print("Unique Values Per Column")
for col in df.columns:
    print(f"{col}: {df[col].nunique()}")

print("\nNumerical Statistics")
display(df.describe())


### What this means

- `nunique()` tells us how many different values occur in each column.
- `describe()` gives statistical information such as **count, mean, standard deviation, minimum, quartiles and maximum**.

The PDF shows that `class` has **2 unique values**, confirming that it is a binary target. The cleaned dataset's summary statistics are based on 4,210 rows. fileciteturn0file0L128-L196


## 6. Separate Features and Target

In [ ]:
# The target is 'class'; all other columns are numeric features.
numeric_features = df.drop(columns=["class"]).select_dtypes(include=np.number).columns.tolist()
categorical_columns = ["class"]

print("Numeric Columns (features only)")
print(numeric_features)

print("\nCategorical Columns (incl. target if set)")
print(categorical_columns)


### What this means

Here, **features (`X`)** are the input variables used to describe an email, while **`class` (`y`)** is the output/target.

Although `class` is stored as an integer, it is treated as a categorical/binary target because its two values represent two classes: non-spam and spam. The PDF explicitly lists `class` separately from the numeric features. fileciteturn0file0L201-L215


## 7. IQR Outlier Detection and Removal

In [ ]:
# IQR = Q3 - Q1
# Lower bound = Q1 - 1.5*IQR
# Upper bound = Q3 + 1.5*IQR

df_iqr = df.copy()

for col in numeric_features:
    q1 = df_iqr[col].quantile(0.25)
    q3 = df_iqr[col].quantile(0.75)
    iqr = q3 - q1

    if iqr == 0:
        print(f"{col}: skipped (IQR is 0, likely a sparse/constant column)")
        continue

    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    outlier_mask = (df_iqr[col] < lower) | (df_iqr[col] > upper)
    print(f"{col}: {outlier_mask.sum()} outliers flagged")

    # Remove rows outside the IQR limits.
    df_iqr = df_iqr.loc[~outlier_mask].copy()

df_clean = df_iqr.reset_index(drop=True)

print("\nFinal shape after IQR filtering:", df_clean.shape)


### Important: what is an outlier?

An **outlier** is a value that is unusually far from the central part of the data.

The **IQR (Interquartile Range)** method uses:

- Q1 = 25th percentile
- Q3 = 75th percentile
- IQR = Q3 − Q1
- Lower limit = Q1 − 1.5 × IQR
- Upper limit = Q3 + 1.5 × IQR

The PDF reports that many sparse columns were skipped because **Q1 = Q3 = 0**, giving IQR = 0. It also reports outliers in several skewed features such as `word_freq_all`, `word_freq_our`, `word_freq_mail`, `word_freq_free`, `word_freq_hp`, `word_freq_re`, and character-frequency columns. fileciteturn0file0L216-L270

**Important note:** the exact row count can depend on the precise implementation of sequential filtering. The PDF's reported final result is **809 rows**, and its discussion says the filtering was applied across 34 skewed columns simultaneously. fileciteturn0file0L277-L283 fileciteturn0file0L307-L311


## 8. Correlation Matrix

In [3]:
plt.figure(figsize=(16, 12))
corr = df_clean[numeric_features].corr()

sns.heatmap(
    corr,
    cmap="coolwarm",
    center=0,
    xticklabels=False,
    yticklabels=False
)

plt.title("Correlation Matrix of Numeric Features")
plt.tight_layout()
plt.show()


NameError: name 'df_clean' is not defined

<Figure size 1600x1200 with 0 Axes>

### How to explain the correlation matrix

**Correlation** measures how two numerical variables change together.

- Close to **+1** → strong positive relationship.
- Close to **−1** → strong negative relationship.
- Close to **0** → little linear relationship.

The PDF says most pairwise correlations are near zero, so **multicollinearity is not a major concern** and most features can be retained. fileciteturn0file0L274-L276 fileciteturn0file0L295-L298


## 9. Distribution of Numeric Features

In [ ]:
# Plot distributions for the numeric feature columns.
# A subset is shown at a time so the notebook remains readable.

for start in range(0, len(numeric_features), 12):
    subset = numeric_features[start:start + 12]
    df_clean[subset].hist(figsize=(16, 12), bins=30)
    plt.suptitle(f"Numeric Feature Distributions ({start+1}-{start+len(subset)})")
    plt.tight_layout()
    plt.show()


### What the distribution plots show

The PDF describes the word-frequency and character-frequency features as **zero-inflated and right-skewed**.

- **Zero-inflated:** many observations are exactly 0.
- **Right-skewed:** most values are small, but a smaller number of observations have much larger values.

This explains why many columns have IQR = 0 and were skipped during the IQR procedure. fileciteturn0file0L287-L289 fileciteturn0file0L299-L302


## 10. Class Distribution

In [ ]:
class_counts = df_clean["class"].value_counts().sort_index()

print(class_counts)

plt.figure(figsize=(6, 4))
sns.countplot(x="class", data=df_clean)
plt.xlabel("Class (0 = non-spam, 1 = spam)")
plt.ylabel("Number of Emails")
plt.title("Class Distribution After Cleaning and Outlier Removal")
plt.show()


### What the class distribution means

`class = 0` means **non-spam** and `class = 1` means **spam**.

The PDF reports **616 non-spam and 193 spam emails**, approximately **76:24**, after its cleaning/outlier-removal process. fileciteturn0file0L290-L291

This is called **class imbalance** because the two classes do not have equal numbers of samples.

For a later spam-classification model, the PDF recommends looking beyond accuracy and considering **precision, recall, F1-score and ROC-AUC**, with possible class weighting or resampling. fileciteturn0file0L303-L306


## 11. Cleaning Summary

| Stage | Rows | Columns |
|---|---:|---:|
| Original | 4601 | 58 |
| After missing-value removal | 4601 | 58 |
| After duplicate removal | 4210 | 58 |
| After IQR outlier removal | 809 | 58 |

These are the values reported in the PDF. fileciteturn0file0L277-L283


## 12. Final Discussion

1. The correlation matrix shows mostly near-zero feature correlations, so multicollinearity is not a major issue.
2. Most word-frequency features are zero-inflated and right-skewed.
3. Many columns were skipped by IQR because their IQR was zero.
4. The final class distribution is moderately imbalanced.
5. IQR filtering caused a very large reduction in the dataset: from 4,210 rows after deduplication to 809 rows in the PDF.
6. The PDF itself notes that a milder filtering strategy could preserve more data for a downstream classifier. fileciteturn0file0L295-L311


## 13. Learning Outcomes

- Use NumPy and Pandas for data handling and manipulation.
- Identify and handle missing values.
- Apply the IQR method for outlier detection/removal.
- Understand the limitations of IQR on zero-inflated, skewed features.
- Interpret distributions, class frequencies and feature correlations.
- Understand why class balance matters when selecting classification metrics.

These learning outcomes follow the experiment report. fileciteturn0file0L312-L322


## Quick Viva / Faculty Explanation

**What is EDA?**  
Exploratory Data Analysis is the process of inspecting, cleaning and visualizing data to understand its structure and important patterns before machine-learning modelling.

**Why did you remove duplicates?**  
To avoid repeated observations influencing the analysis more than they should.

**Why were there no missing-value changes?**  
The dataset already contained zero missing values in all 58 columns.

**What is IQR?**  
IQR is Q3 − Q1. Values below Q1 − 1.5×IQR or above Q3 + 1.5×IQR are treated as outliers.

**Why were some columns skipped?**  
Their IQR was zero because the data was highly sparse/zero-inflated, so the standard IQR rule could not meaningfully identify outliers.

**What does class mean?**  
0 = non-spam and 1 = spam.

**Why is class imbalance important?**  
If one class is much larger, accuracy alone can hide poor performance on the smaller class. Precision, recall, F1-score and ROC-AUC are more informative.

**What is the main conclusion?**  
The Spambase data is numerical, sparse and skewed, has little problematic pairwise correlation, contains duplicate rows and shows class imbalance. The report concludes that careful preprocessing is important before building a spam classifier.
